# Cache Line & False Sharing Mechanics in Multi-Core Python
**Instructor:** Sufyan bin Uzayr · NumPy + Numba `@njit(nogil=True)` + `threading`

⚠️ Best with **≥ 4 logical CPU cores** (no GPU needed). On a 2-core Colab runtime it still runs, oversubscribed.
Use a runtime with more cores: **Runtime → Change runtime type → TPU (v2-8 / v5e)** or **High-RAM** (8 cores, Pro),
or run `python3 false_sharing_lab.py <username>` on your own laptop.

**Student:** Iliyas Malibekov · **ID:** 230103344 · **GitHub:** 230103344

### 1 — Check CPU topology

In [1]:
import os
!lscpu | grep -E "Model name|^CPU\(s\)|Thread\(s\) per core|Core\(s\) per socket|L1d|L2"
!cat /sys/devices/system/cpu/cpu0/cache/index0/coherency_line_size
print("os.cpu_count() =", os.cpu_count())
if (os.cpu_count() or 1) < 4:
    print("\n⚠️ Less than 4 cores: the lab still runs (4 threads oversubscribed), but false sharing is weaker.")
    print("   For clean numbers use a TPU / High-RAM runtime or your Mac.")
else:
    print("\n✅ Enough cores.")
!pip install -q numba numpy

CPU(s):                                  2
Model name:                              AMD EPYC 7B12
Thread(s) per core:                      2
Core(s) per socket:                      1
L1d cache:                               32 KiB (1 instance)
L2 cache:                                512 KiB (1 instance)
64
os.cpu_count() = 2

⚠️ Less than 4 cores: the lab still runs (4 threads oversubscribed), but false sharing is weaker.
   For clean numbers use a TPU / High-RAM runtime or your Mac.


### 2 — Your Git username

In [2]:
GIT_USERNAME = "230103344"
assert GIT_USERNAME != "REPLACE_WITH_YOUR_GIT_USERNAME" and GIT_USERNAME.strip(), "Set your real Git handle!"
print("Git identity:", GIT_USERNAME)

Git identity: 230103344


### 3 — `false_sharing_lab.py` (Tasks 1–5)

In [3]:
%%writefile false_sharing_lab.py
import os
import sys
import json
import time
import ctypes
import hashlib
import platform
import threading
import subprocess
import statistics
import numpy as np
from numba import njit, uint64

try:
    import resource
except ImportError:
    resource = None

# Replace with your exact Git handle before executing
# (can also be passed as: python3 false_sharing_lab.py <git_username>)
GIT_USERNAME = "230103344"
NUM_THREADS = 4
ITERATIONS_PER_THREAD = 60_000_000
STRIDES = (1, 4, 8, 16)
REPEATS = 3                      # each layout is timed REPEATS times, median is reported
ALIGNMENT = 128                  # 128 B alignment => also 64 B aligned, and slot 0 starts a 128 B pair
GIL_CONTROL_ITERS = 1_000_000    # pure-Python control for Q5 (kept small: bytecode is slow)


def detect_cache_line_size() -> int:
    """Query L1 data cache coherency line size in bytes from OS sysfs/sysctl."""
    linux_path = "/sys/devices/system/cpu/cpu0/cache/index0/coherency_line_size"
    if os.path.exists(linux_path):
        with open(linux_path, "r", encoding="utf-8") as f:
            return int(f.read().strip())
    if sys.platform == "darwin":
        out = subprocess.check_output(["sysctl", "-n", "hw.cachelinesize"], encoding="utf-8")
        return int(out.strip())
    return 64


def detect_cpu_model() -> str:
    try:
        if sys.platform.startswith("linux"):
            with open("/proc/cpuinfo", encoding="utf-8") as f:
                for line in f:
                    if line.lower().startswith(("model name", "hardware", "cpu model")):
                        return line.split(":", 1)[1].strip()
        if sys.platform == "darwin":
            return subprocess.check_output(["sysctl", "-n", "machdep.cpu.brand_string"],
                                           encoding="utf-8").strip()
    except Exception:
        pass
    return platform.processor() or platform.machine()


def pick_physical_cores(available, k):
    """Prefer one logical CPU per physical core (avoid SMT siblings sharing one L1)."""
    chosen, seen = [], set()
    for cpu in sorted(available):
        path = f"/sys/devices/system/cpu/cpu{cpu}/topology/core_id"
        pkg = f"/sys/devices/system/cpu/cpu{cpu}/topology/physical_package_id"
        try:
            key = (open(pkg).read().strip(), open(path).read().strip())
        except OSError:
            key = ("?", str(cpu))
        if key not in seen:
            seen.add(key)
            chosen.append(cpu)
        if len(chosen) == k:
            return chosen
    # not enough distinct physical cores -> fill with remaining logical CPUs (cycling if needed)
    for cpu in sorted(available):
        if cpu not in chosen and len(chosen) < k:
            chosen.append(cpu)
    i = 0
    avail = sorted(available)
    while len(chosen) < k:
        chosen.append(avail[i % len(avail)])
        i += 1
    return chosen


# ==============================================================================
# TASK 1: Hardware Topology & Student Identity Context
# ==============================================================================
def init_system_context(git_user: str) -> dict:
    assert git_user != "REPLACE_WITH_YOUR_GIT_USERNAME" and len(git_user.strip()) > 0, (
        "Set GIT_USERNAME to your real Git handle before running."
    )
    logical_cores = os.cpu_count() or 1
    oversubscribed = logical_cores < NUM_THREADS
    if oversubscribed:
        # Free Colab CPU runtimes expose only 2 vCPUs. We still run T = 4 threads, but they
        # time-share the cores; this is recorded in the manifest ("oversubscribed": true).
        print(f"WARNING: only {logical_cores} logical cores (< {NUM_THREADS}); running oversubscribed.")

    line_size = detect_cache_line_size()
    cpu_sig = f"{platform.machine()}::{platform.processor()}::{logical_cores}"

    # Deterministic 64-bit seed: SHA-256(git_user::cpu_sig), first 16 hex digits
    digest = hashlib.sha256(f"{git_user.strip()}::{cpu_sig}".encode("utf-8")).hexdigest()
    instance_seed = int(digest[:16], 16)

    # Core affinity (Linux only; macOS has no user-level thread pinning API)
    affinity_supported = hasattr(os, "sched_setaffinity") and hasattr(os, "sched_getaffinity")
    if affinity_supported:
        available = sorted(os.sched_getaffinity(0))
        pinned_cores = pick_physical_cores(available, NUM_THREADS)
    else:
        available = list(range(logical_cores))
        pinned_cores = [None] * NUM_THREADS

    ctx = dict(
        git_user=git_user.strip(),
        logical_cores=logical_cores,
        cpu_model=detect_cpu_model(),
        cpu_sig=cpu_sig,
        os=f"{platform.system()} {platform.release()}",
        python=platform.python_version(),
        cache_line_size=line_size,
        instance_seed=instance_seed,
        sha256=digest,
        affinity_supported=affinity_supported,
        oversubscribed=oversubscribed,
        available_cpus=available,
        pinned_cores=pinned_cores,
    )
    print("=" * 78)
    print(f"Git user          : {ctx['git_user']}")
    print(f"CPU model         : {ctx['cpu_model']}")
    print(f"CPU signature     : {cpu_sig}")
    print(f"Logical cores     : {logical_cores}")
    print(f"Cache line size   : {line_size} B")
    print(f"INSTANCE_SEED     : {instance_seed} (0x{instance_seed:016x})")
    print(f"Affinity pinning  : {'ON -> cores ' + str(pinned_cores) if affinity_supported else 'not supported by OS'}")
    print("=" * 78, flush=True)
    return ctx


# ==============================================================================
# TASK 2: GIL-Free Native Counter Kernel
# ==============================================================================
@njit(nogil=True, cache=False)
def worker_strided_accumulate(buffer, slot, iters, seed):
    """In-place RMW on buffer[slot] every iteration (no register promotion).

    The store address is re-derived each iteration from a value LOADED from the
    same buffer (idx = slot + (buffer[slot] >> 63)). At run time the shift is
    always 0 (counters never reach 2**63), but LLVM cannot prove that, so the
    address is not loop-invariant and the store may alias the next load =>
    the counter cannot be kept in a register; every iteration does a real
    load + store to L1.
    """
    s = uint64(seed)
    for i in range(iters):
        v = buffer[slot]
        idx = slot + (v >> uint64(63))
        buffer[idx] = buffer[idx] + (uint64(1) + ((s + uint64(i)) & uint64(3)))
    return buffer[slot]


@njit(nogil=True, cache=False)
def worker_naive_hoistable(buffer, slot, iters, seed):
    """Reference ONLY for the LLVM audit (Q4): LLVM promotes buffer[slot] to a register here."""
    s = uint64(seed)
    for i in range(iters):
        buffer[slot] = buffer[slot] + (uint64(1) + ((s + uint64(i)) & uint64(3)))
    return buffer[slot]


# ==============================================================================
# TASK 4: Thread-Local Register Privatization Baseline
# ==============================================================================
@njit(nogil=True, cache=False)
def worker_private_accumulate(buffer, slot, iters, seed):
    """Accumulate in a local scalar (register); single store to shared memory at the end."""
    s = uint64(seed)
    acc = buffer[slot]
    for i in range(iters):
        acc = acc + (uint64(1) + ((s + uint64(i)) & uint64(3)))
    buffer[slot] = acc
    return acc


def py_strided_accumulate(buffer, slot, iters, seed):
    """Pure CPython bytecode version (holds the GIL) — Q5 control experiment."""
    for i in range(iters):
        buffer[slot] = buffer[slot] + 1 + ((seed + i) & 3)


# ------------------------------------------------------------------------------
def aligned_uint64_buffer(n_elems: int, alignment: int = ALIGNMENT) -> np.ndarray:
    raw = np.zeros(n_elems * 8 + alignment, dtype=np.uint8)
    offset = (-raw.ctypes.data) % alignment
    buf = raw[offset: offset + n_elems * 8].view(np.uint64)
    assert buf.ctypes.data % alignment == 0 and buf.flags["C_CONTIGUOUS"]
    return buf


def thread_seed(ctx, t):
    return (ctx["instance_seed"] + 0x9E3779B97F4A7C15 * (t + 1)) & 0xFFFFFFFFFFFFFFFF


def expected_counter(seed: int, iters: int) -> int:
    """Closed form of sum_{i<iters} (1 + ((seed+i) & 3)) mod 2**64 (independent check)."""
    full, rem = divmod(iters, 4)
    total = iters + full * (0 + 1 + 2 + 3)
    for i in range(rem):
        total += (seed + full * 4 + i) & 3
    return total & 0xFFFFFFFFFFFFFFFF


def rusage_snapshot():
    if resource is None:
        return (0, 0)
    r = resource.getrusage(resource.RUSAGE_SELF)
    return (r.ru_nvcsw, r.ru_nivcsw)


def _run_threads(ctx, kernel, buffer, slots, iters, pin=True, python_mode=False):
    """Launch NUM_THREADS threads behind a barrier; return elapsed ns + ctx-switch deltas."""
    n = len(slots)
    barrier = threading.Barrier(n + 1)
    errors = []

    def body(t):
        try:
            core = ctx["pinned_cores"][t] if pin else None
            if core is not None:
                os.sched_setaffinity(0, {core})          # 0 = calling thread on Linux
            seed = thread_seed(ctx, t)
            seed = int(seed) if python_mode else np.uint64(seed)
            barrier.wait()
            kernel(buffer, slots[t], iters, seed)
        except Exception as e:                            # pragma: no cover
            errors.append(repr(e))
            barrier.abort()

    threads = [threading.Thread(target=body, args=(t,), name=f"worker-{t}") for t in range(n)]
    for th in threads:
        th.start()
    v0, iv0 = rusage_snapshot()
    try:
        barrier.wait()
    except threading.BrokenBarrierError:
        pass
    t0 = time.perf_counter_ns()
    for th in threads:
        th.join()
    elapsed = time.perf_counter_ns() - t0
    v1, iv1 = rusage_snapshot()
    if errors:
        raise RuntimeError(errors)
    return elapsed, v1 - v0, iv1 - iv0


# ==============================================================================
# TASK 3: Multi-Threaded Stride Sweep
# ==============================================================================
def run_stride_benchmark(ctx) -> dict:
    iters = ITERATIONS_PER_THREAD
    total_updates = NUM_THREADS * iters
    line = ctx["cache_line_size"]
    results = {}

    # JIT warm-up (compilation excluded from timing)
    wb = aligned_uint64_buffer(16)
    for k in (worker_strided_accumulate, worker_private_accumulate, worker_naive_hoistable):
        k(wb, 0, 1000, np.uint64(1))      # same arg types as the timed calls -> no re-JIT

    configs = [(f"stride_{s}", s, worker_strided_accumulate) for s in STRIDES]
    configs.append(("private_register", 8, worker_private_accumulate))

    print(f"\n{'layout':18s} {'stride':>6s} {'byte offsets':>22s} {'lines':>12s} "
          f"{'median ns':>16s} {'ns/update':>10s} {'vcsw':>6s} {'ivcsw':>6s}")
    for name, stride, kernel in configs:
        n_elems = stride * NUM_THREADS + 32
        slots = [t * stride for t in range(NUM_THREADS)]
        runs, vcs, ivcs, finals = [], [], [], []
        for _ in range(REPEATS):
            buf = aligned_uint64_buffer(n_elems)
            el, vc, ivc = _run_threads(ctx, kernel, buf, slots, iters)
            runs.append(el)
            vcs.append(vc)
            ivcs.append(ivc)
            finals.append([int(buf[s]) for s in slots])
        base = buf.ctypes.data
        offsets = [s * 8 for s in slots]
        line_idx = [(base + o) // line - base // line for o in offsets]
        pair_idx = [(base + o) // 128 - base // 128 for o in offsets]
        med = int(statistics.median(runs))
        results[name] = dict(
            stride=stride,
            kernel=kernel.__name__ if hasattr(kernel, "__name__") else str(kernel),
            byte_offsets=offsets,
            cache_line_index=line_idx,
            prefetch_pair_128B_index=pair_idx,
            runs_ns=runs,
            median_ns=med,
            min_ns=int(min(runs)),
            ns_per_update=med / total_updates,
            ns_per_update_per_thread=med / iters,
            voluntary_ctx_switches=vcs,
            involuntary_ctx_switches=ivcs,
            final_counters=finals[-1],
            all_repeats_identical=all(f == finals[0] for f in finals),
        )
        print(f"{name:18s} {stride:6d} {str(offsets):>22s} {str(line_idx):>12s} "
              f"{med:16,d} {med / total_updates:10.3f} {statistics.median(vcs):6.0f} "
              f"{statistics.median(ivcs):6.0f}", flush=True)

    ref = results["stride_8"]["median_ns"]
    for name in results:
        results[name]["slowdown_vs_stride_8"] = results[name]["median_ns"] / ref
    print("\nSlowdown vs stride=8: " + ", ".join(
        f"{k}={v['slowdown_vs_stride_8']:.2f}x" for k, v in results.items()))
    return results


def run_gil_control(ctx) -> dict:
    """Q5: identical stride=1 workload in pure Python threads (GIL-serialized)."""
    out = {}
    iters = GIL_CONTROL_ITERS
    for n in (1, NUM_THREADS):
        buf = [0] * (NUM_THREADS * 1 + 8)                    # python list, stride 1
        slots = list(range(n))
        sub = dict(ctx, pinned_cores=ctx["pinned_cores"][:n])
        el, vc, ivc = _run_threads(sub, py_strided_accumulate, buf, slots, iters,
                                   python_mode=True)
        out[f"python_threads_{n}"] = dict(threads=n, iters_per_thread=iters, elapsed_ns=el,
                                          ns_per_update=el / (n * iters),
                                          voluntary_ctx_switches=vc, involuntary_ctx_switches=ivc)
    r = out[f"python_threads_{NUM_THREADS}"]["elapsed_ns"] / out["python_threads_1"]["elapsed_ns"]
    out["wall_ratio_4_vs_1"] = r
    print(f"\n[GIL control] pure-Python stride=1, {iters:,} updates/thread: "
          f"1 thread = {out['python_threads_1']['elapsed_ns']:,} ns, "
          f"{NUM_THREADS} threads = {out[f'python_threads_{NUM_THREADS}']['elapsed_ns']:,} ns "
          f"({r:.2f}x wall time for {NUM_THREADS}x work -> no parallel speedup)")
    return out


# ==============================================================================
# TASK 5: Exact State Parity & LLVM IR Verification
# ==============================================================================
def _native_llvm_function(llvm: str) -> str:
    """Body of the nopython function only (skip the CPython wrapper / cfunc / NRT helpers)."""
    out, keep = [], False
    for line in llvm.splitlines():
        if line.startswith("define "):
            name = line.split("@", 1)[1].split("(", 1)[0]
            keep = name.startswith("_ZN") and "cpython" not in name and "NRT" not in name
        if keep:
            out.append(line)
            if line.startswith("}"):
                keep = False
    return "\n".join(out)


def _native_asm_function(asm: str) -> str:
    out, keep = [], False
    for line in asm.splitlines():
        st = line.strip()
        if not line.startswith((" ", "\t")) and st.endswith(":") and st.startswith("_ZN"):
            keep = "cpython" not in st
        if keep:
            if st.startswith((".loc", ".cfi", ".file")):
                continue
            out.append(line)
            if st.startswith(".Lfunc_end") or st.startswith("Lfunc_end"):
                keep = False
    return "\n".join(out)


def _ir_stats(dispatcher):
    llvm_all = "\n".join(dispatcher.inspect_llvm().values())
    asm_all = "\n".join(dispatcher.inspect_asm().values())
    native_ir = _native_llvm_function(llvm_all)
    return dict(
        store_i64=native_ir.count("store i64"),
        load_i64=native_ir.count("load i64"),
        pyobject_refs_in_native=sum(native_ir.count(k) for k in ("PyObject", "@Py", "pyobject")),
        releases_gil_in_wrapper=("PyEval_SaveThread" in llvm_all and "PyEval_RestoreThread" in llvm_all),
        nopython_signatures=[str(s) for s in dispatcher.nopython_signatures],
        llvm_text=native_ir,
        asm_text=_native_asm_function(asm_all),
    )


def _store_in_loop(asm_text: str) -> bool:
    """x86/ARM heuristic: is there a memory-destination store inside a loop body?"""
    import re
    lines = asm_text.splitlines()
    labels = {l.strip()[:-1]: i for i, l in enumerate(lines) if re.match(r"^\s*\.?L\w+:$", l)}
    for i, l in enumerate(lines):
        m = re.search(r"\b(j\w+|b\.\w+|b|cbnz|cbz|tbnz)\s+(\.?L\w+)", l)
        if m and m.group(2) in labels and labels[m.group(2)] < i:       # backward branch = loop
            body = lines[labels[m.group(2)]: i + 1]
            for b in body:
                if re.search(r"(mov\w*|add\w*)\s+%?\w+,\s*-?\d*\(", b) or re.search(r"\bstr\s", b) \
                        or re.search(r"\bstp?\s", b) or re.search(r"\b(add|mov)q?\s+.*,\s*\(%", b):
                    return True
    return False


def audit_llvm_and_parity(ctx, results, gil) -> dict:
    iters = ITERATIONS_PER_THREAD
    expected = [expected_counter(thread_seed(ctx, t), iters) for t in range(NUM_THREADS)]
    parity = {name: (r["final_counters"] == expected and r["all_repeats_identical"])
              for name, r in results.items()}
    parity_ok = all(parity.values())

    strided = _ir_stats(worker_strided_accumulate)
    naive = _ir_stats(worker_naive_hoistable)
    private = _ir_stats(worker_private_accumulate)
    strided_store_in_loop = _store_in_loop(strided["asm_text"])
    naive_store_in_loop = _store_in_loop(naive["asm_text"])
    with open("worker_strided_accumulate.ll", "w") as f:
        f.write(strided["llvm_text"])
    with open("worker_strided_accumulate.s", "w") as f:
        f.write(strided["asm_text"])

    llvm_ok = strided["store_i64"] > 0 and strided["releases_gil_in_wrapper"] and strided["pyobject_refs_in_native"] == 0 and \
        len(strided["nopython_signatures"]) > 0

    print("\n[Parity] expected final counters per thread:", expected)
    for k, v in parity.items():
        print(f"  {k:18s}: {'BIT-EXACT' if v else 'MISMATCH'}  {results[k]['final_counters']}")
    print(f"[LLVM] worker_strided_accumulate: store i64 = {strided['store_i64']}, "
          f"load i64 = {strided['load_i64']}, PyObject refs in native fn = {strided['pyobject_refs_in_native']}, "
          f"store inside loop (asm) = {strided_store_in_loop}, "
          f"GIL released (PyEval_SaveThread) = {strided['releases_gil_in_wrapper']}")
    print(f"[LLVM] worker_naive_hoistable   : store i64 = {naive['store_i64']}, "
          f"store inside loop (asm) = {naive_store_in_loop}  (promoted to register => False expected)")
    print(f"[LLVM] worker_private_accumulate: store i64 = {private['store_i64']}")

    verification_passed = bool(parity_ok and llvm_ok)
    manifest = dict(
        git_username=ctx["git_user"],
        instance_seed=str(ctx["instance_seed"]),
        instance_seed_hex=f"0x{ctx['instance_seed']:016x}",
        cpu_model=ctx["cpu_model"],
        cpu_signature=ctx["cpu_sig"],
        os=ctx["os"],
        python=ctx["python"],
        numba=__import__("numba").__version__,
        numpy=np.__version__,
        logical_cores=ctx["logical_cores"],
        oversubscribed=ctx["oversubscribed"],
        cache_line_size=ctx["cache_line_size"],
        affinity_supported=ctx["affinity_supported"],
        pinned_cores=ctx["pinned_cores"],
        num_threads=NUM_THREADS,
        iterations_per_thread=ITERATIONS_PER_THREAD,
        repeats=REPEATS,
        buffer_alignment_bytes=ALIGNMENT,
        results=results,
        slowdown_stride1_vs_stride8=results["stride_1"]["slowdown_vs_stride_8"],
        slowdown_stride4_vs_stride8=results["stride_4"]["slowdown_vs_stride_8"],
        stride16_vs_stride8=results["stride_16"]["slowdown_vs_stride_8"],
        private_vs_stride8=results["private_register"]["slowdown_vs_stride_8"],
        gil_control=gil,
        expected_counters=[str(e) for e in expected],
        parity=parity,
        parity_passed=parity_ok,
        llvm_audit=dict(
            worker_strided_accumulate=dict(
                store_i64=strided["store_i64"], load_i64=strided["load_i64"],
                pyobject_refs_in_native=strided["pyobject_refs_in_native"],
                releases_gil_in_wrapper=strided["releases_gil_in_wrapper"],
                nopython_signatures=strided["nopython_signatures"],
                store_inside_loop_asm=strided_store_in_loop),
            worker_naive_hoistable=dict(
                store_i64=naive["store_i64"], store_inside_loop_asm=naive_store_in_loop),
            worker_private_accumulate=dict(store_i64=private["store_i64"]),
            passed=llvm_ok,
        ),
        timestamp_utc=time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        verification_passed=verification_passed,
    )
    with open("cache_lab_manifest.json", "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)
    print(f"\ncache_lab_manifest.json written | verification_passed = {verification_passed}")
    return manifest


def main():
    git_user = sys.argv[1] if len(sys.argv) > 1 else os.environ.get("GIT_USERNAME", GIT_USERNAME)
    ctx = init_system_context(git_user)
    results = run_stride_benchmark(ctx)
    gil = run_gil_control(ctx)
    manifest = audit_llvm_and_parity(ctx, results, gil)
    if not manifest["verification_passed"]:
        sys.exit(1)


if __name__ == "__main__":
    main()


Writing false_sharing_lab.py


### 4 — Run the full benchmark suite

In [4]:
import subprocess, sys
proc = subprocess.run([sys.executable, "false_sharing_lab.py", GIT_USERNAME], capture_output=True, text=True)
print(proc.stdout)
if proc.returncode != 0:
    print("STDERR:\n", proc.stderr)
    raise RuntimeError("verification failed / script error")

Git user          : 230103344
CPU model         : AMD EPYC 7B12
CPU signature     : x86_64::x86_64::2
Logical cores     : 2
Cache line size   : 64 B
INSTANCE_SEED     : 17949844368381285551 (0xf91aa859c6e994af)
Affinity pinning  : ON -> cores [0, 1, 0, 1]

layout             stride           byte offsets        lines        median ns  ns/update   vcsw  ivcsw
stride_1                1         [0, 8, 16, 24] [0, 0, 0, 0]      646,768,429      2.695      6    540
stride_4                4        [0, 32, 64, 96] [0, 0, 1, 1]      441,062,168      1.838      5    545
stride_8                8      [0, 64, 128, 192] [0, 1, 2, 3]      560,591,247      2.336      5    321
stride_16              16     [0, 128, 256, 384] [0, 2, 4, 6]      227,506,768      0.948      7    240
private_register        8      [0, 64, 128, 192] [0, 1, 2, 3]       13,521,059      0.056      7     16

Slowdown vs stride=8: stride_1=1.15x, stride_4=0.79x, stride_8=1.00x, stride_16=0.41x, private_register=0.02x

[GIL co

### 5 — Numbers for ANALYSIS.md

In [5]:
import json
m = json.load(open("cache_lab_manifest.json"))
print("verification_passed:", m["verification_passed"])
print("CPU:", m["cpu_model"], "| cores:", m["logical_cores"], "| line:", m["cache_line_size"], "B | pinned:", m["pinned_cores"])
print("seed:", m["instance_seed_hex"])
print(f"\n{'layout':18s}{'median ns':>16s}{'ns/update':>11s}{'ns/upd/thread':>15s}{'slowdown':>10s}  lines")
for k, r in m["results"].items():
    print(f"{k:18s}{r['median_ns']:16,d}{r['ns_per_update']:11.3f}{r['ns_per_update_per_thread']:15.3f}"
          f"{r['slowdown_vs_stride_8']:9.2f}x  {r['cache_line_index']}")
print("\nGIL control:", {k: v for k, v in m["gil_control"].items()})
print("\nLLVM audit:", json.dumps(m["llvm_audit"], indent=1))

verification_passed: True
CPU: AMD EPYC 7B12 | cores: 2 | line: 64 B | pinned: [0, 1, 0, 1]
seed: 0xf91aa859c6e994af

layout                   median ns  ns/update  ns/upd/thread  slowdown  lines
stride_1               646,768,429      2.695         10.779     1.15x  [0, 0, 0, 0]
stride_4               441,062,168      1.838          7.351     0.79x  [0, 0, 1, 1]
stride_8               560,591,247      2.336          9.343     1.00x  [0, 1, 2, 3]
stride_16              227,506,768      0.948          3.792     0.41x  [0, 2, 4, 6]
private_register        13,521,059      0.056          0.225     0.02x  [0, 1, 2, 3]

GIL control: {'python_threads_1': {'threads': 1, 'iters_per_thread': 1000000, 'elapsed_ns': 118066498, 'ns_per_update': 118.066498, 'voluntary_ctx_switches': 1, 'involuntary_ctx_switches': 5}, 'python_threads_4': {'threads': 4, 'iters_per_thread': 1000000, 'elapsed_ns': 494488543, 'ns_per_update': 123.62213575, 'voluntary_ctx_switches': 406, 'involuntary_ctx_switches': 137}, 

In [6]:
# Proof for Q4: memory store inside the hot loop (x86 asm of the native function)
!grep -nE "^\.LBB|add[q]? .*\(|mov[q]? .*,.*\(|j[a-z]+ +\.LBB" worker_strided_accumulate.s | head -40

17:.LBB0_2:
32:.LBB0_3:


### 6 — ANALYSIS.md template (fill the ⟨…⟩ with your numbers, then download)

In [7]:
%%writefile ANALYSIS.md
# ANALYSIS — Cache Line & False Sharing Mechanics

**Student:** Iliyas Malibekov · **ID:** 230103344 · **GitHub:** 230103344 · **Repo:** cache-line-false-sharing-230103344

> Replace every `⟨…⟩` with the values from **your** `cache_lab_manifest.json` / stdout.

**Host:** ⟨cpu_model⟩ · ⟨logical_cores⟩ logical cores · cache line = ⟨cache_line_size⟩ B · pinned cores = ⟨pinned_cores⟩
**INSTANCE_SEED:** ⟨instance_seed_hex⟩ · **T = 4 threads × 60,000,000 updates** (240,000,000 updates per layout, median of 3 runs)

| Layout | stride | byte offsets | cache lines | median ns | ns / update | slowdown vs stride 8 |
|---|---|---|---|---|---|---|
| A Contiguous | 1 | 0, 8, 16, 24 | 0,0,0,0 | ⟨…⟩ | ⟨…⟩ | ⟨…⟩× |
| B Half-line | 4 | 0, 32, 64, 96 | 0,0,1,1 | ⟨…⟩ | ⟨…⟩ | ⟨…⟩× |
| C Line-padded | 8 | 0, 64, 128, 192 | 0,1,2,3 | ⟨…⟩ | ⟨…⟩ | 1.00× |
| D Prefetch-safe | 16 | 0, 128, 256, 384 | 0,2,4,6 | ⟨…⟩ | ⟨…⟩ | ⟨…⟩× |
| Private register | — | 0, 64, 128, 192 | (1 store) | ⟨…⟩ | ⟨…⟩ | ⟨…⟩× |

## Q1 — MESI Protocol State Transitions (stride = 1)
`buffer[0]` and `buffer[1]` are 8 B apart, so they sit in the **same 64-byte line L**.
1. Start: L is in no cache, i.e. **I/I** (Core 0 / Core 1).
2. Core 0 stores to `buffer[0]`. This is a write miss, so Core 0 issues an **RFO** (Read-For-Ownership = read + invalidate). No other core holds L, so Core 0 gets it exclusively, **I→E**, and the store makes it **E→M**. State: **M/I**.
3. Core 1 stores to `buffer[1]`. Its copy is Invalid, so it puts the store into its **store buffer** and sends an RFO for L. Core 0 snoops the RFO, writes back/forwards its dirty line (**M→I**), and Core 1 receives the line, **I→M**. State: **I/M**.
4. Core 0's next increment of `buffer[0]` needs L again: it sends an RFO, so Core 1 goes **M→I** and Core 0 goes **I→M**. The ping-pong repeats for every iteration on every core.

There is no `Shared` state for long. Both cores only *write*, so every request is an RFO that invalidates the other copy. Core 1's store **cannot retire** from the store buffer into L1 until it owns L in M/E state. The store buffer fills up while each RFO waits a full cross-core round trip (tens of ns through L3/the ring/mesh instead of ~1 ns L1 hits). Our kernel also *loads* `buffer[slot]` each iteration (read-modify-write), so the dependent load also stalls on the same line. With 4 threads on one line, ownership rotates through 4 cores, which is the **4-way RFO ping-pong** in my stride-1 timing (⟨…⟩ ns/update).

## Q2 — Quantitative Slowdown & Coherency Overhead
- stride 1: ⟨…⟩ ns total → ⟨…⟩ ns per update (240 M updates) → ⟨…⟩ ns per update per thread
- stride 4: ⟨…⟩ ns → ⟨…⟩ ns/update
- stride 8: ⟨…⟩ ns → ⟨…⟩ ns/update
- stride 16: ⟨…⟩ ns → ⟨…⟩ ns/update
- private register: ⟨…⟩ ns → ⟨…⟩ ns/update

**Slowdown stride 1 / stride 8 = ⟨…⟩×.** The stride-8 time is the cost of an L1-resident load→add→store chain. That chain is limited by **store-to-load forwarding** latency, because each iteration reloads the value it just stored (≈4–5 cycles ≈ ⟨…⟩ ns per update per thread). The **extra** ⟨stride1 − stride8⟩ ns per update is pure **cross-core coherence latency** (RFO + invalidation). Stride 4 (⟨…⟩×) is in between: only 2 cores fight over each line (2-way false sharing). The privatized baseline (⟨…⟩×) keeps the counter in a register, so it has neither the memory chain nor coherence traffic. Comparing private vs stride 8 isolates the store-forwarding cost; comparing stride 8 vs stride 1 isolates the coherence cost. Context switches (vcsw/ivcsw ≈ ⟨…⟩) are tiny for all layouts, so the slowdown is **not** caused by the OS scheduler. It comes from hardware.

## Q3 — L2 Spatial Prefetcher & 128-Byte Pairing
stride 8 = ⟨…⟩ ns vs stride 16 = ⟨…⟩ ns (ratio ⟨…⟩×). Intel's **L2 adjacent-cache-line (spatial) prefetcher** treats memory as 128-byte-aligned pairs of 64-byte lines (even line 2k + odd line 2k+1). When a core misses on one line, it also fetches the "buddy" line. With stride 8, threads 0/1 own lines 0/1, which are the *same 128-B pair*, and threads 2/3 own lines 2/3. A miss or RFO by thread 0 can make its core prefetch line 1, which pulls a copy of thread 1's line into the wrong core. The next store by thread 1 then has to invalidate it again. This is a **secondary invalidation** even though no two threads share a 64-B line. It happens when the prefetcher is enabled, the threads are on different cores, the lines are in the same 128-B-aligned pair, and the lines miss often (e.g. after other coherence traffic or context migration). With stride 16 every thread owns a different 128-B pair, so the prefetcher can't create cross-thread traffic. ⟨If your stride-16 is only slightly faster or equal: in steady state each line stays in M in its owner's L1, so there are few misses and the prefetcher rarely triggers; the effect is small.⟩ (On Apple Silicon `hw.cachelinesize` = 128 B, so stride 8 is real 2-way false sharing there and only stride 16 is padded.)

## Q4 — Compiler Register Promotion vs. Volatile Memory Stores
LLVM's **LICM / scalar promotion** pass moves a memory location into a register when (a) its address is loop-invariant and (b) no other access in the loop may alias it. In a naive loop `buffer[slot] += inc`, both hold. LLVM loads `buffer[slot]` once, keeps it in a register, and stores it once after the loop. Single-threaded semantics allow this because Numba arrays are not `volatile`/atomic. My audit proves it. In `worker_naive_hoistable` the asm loop has **no store inside the loop** (`store_inside_loop_asm = ⟨false⟩`): it is even vectorized into `vpaddq` and stored once after the loop. That loop would show no false sharing at all.

In `worker_strided_accumulate` the store address is computed each iteration from a **loaded value**:
```python
v = buffer[slot]; idx = slot + (v >> 63); buffer[idx] = buffer[idx] + inc
```
Because `idx` depends on memory, the address is **not loop-invariant**. The store to `buffer[idx]` *may alias* the next load of `buffer[slot]`, so LLVM must reload and store every iteration. At run time `v >> 63` is always 0 (counters stay ≪ 2⁶³), so the result is identical. The native function's IR has **⟨…⟩ `store i64`** and **⟨…⟩ `load i64`**, with **⟨0⟩ PyObject references** (pure nopython). The signature is `⟨(Array(uint64, 1, 'C', …), int64, int64, uint64) -> uint64⟩`. In the asm the loop body contains a memory-destination add (`addq %reg, (%base,%idx,8)` on x86), so **every iteration issues a real L1 store** (`store_inside_loop_asm = ⟨true⟩`).

## Q5 — CPython GIL Masking vs. `nogil=True`
Pure-Python stride-1 control (⟨…⟩ updates/thread): 1 thread = ⟨…⟩ ns, 4 threads = ⟨…⟩ ns (⟨…⟩× wall time for 4× work). This means **zero parallel speedup**. Only one thread can execute bytecode while holding the GIL, so the 4 threads never store to the cache line *at the same time*. There is no simultaneous RFO ping-pong, so false sharing cannot appear. Each `buffer[slot] = buffer[slot] + …` is dozens of bytecodes: refcounting, boxed int allocation, list item writes. That is hundreds of ns per update, so any coherence miss is hidden. More threads make it **slower**, because of GIL **lock contention**: the interpreter forces a switch every 5 ms (`sys.getswitchinterval`), threads wait on the GIL mutex/condvar (context switches: ⟨…⟩), and the GIL itself and interpreter state bounce between cores. That is *software* lock contention, not hardware cache-line contention. With `@njit(nogil=True)` the wrapper calls `PyEval_SaveThread()` before the native loop (`releases_gil_in_wrapper = ⟨true⟩` in my manifest) and `PyEval_RestoreThread()` after. The 4 threads then run machine code truly in parallel on 4 cores, and only then does the hardware MESI contention appear (stride 1 = ⟨…⟩× slower than stride 8).


Writing ANALYSIS.md


In [8]:
from google.colab import files
for f in ["false_sharing_lab.py", "cache_lab_manifest.json", "ANALYSIS.md"]:
    files.download(f)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>